In [1]:
!apt-get install tesseract-ocr -y
!pip install pytesseract pdf2image pillow opencv-python pandas matplotlib

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path
from PIL import Image
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab import files
import re

In [30]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving HCFA1500.jpg to HCFA1500 (1).jpg


In [31]:
def load_images(path):

    images = []

    if path.lower().endswith(".pdf"):
        pages = convert_from_path(path)
        images.extend(pages)

    else:
        img = Image.open(path)

        try:
            while True:
                images.append(img.copy())
                img.seek(len(images))
        except EOFError:
            pass

    return images

images = load_images(file_path)

In [32]:
def get_ocr_dataframe(image):

    df = pytesseract.image_to_data(
        image,
        output_type=Output.DATAFRAME
    )

    df = df.dropna(subset=["text"])
    df = df[df["text"].str.strip() != ""]

    return df

ocr_df = get_ocr_dataframe(images[0])

In [33]:
FIELD_CONFIG = {

    "patient_name": {
        "anchor": "PATIENT",
        "type": "text"
    },

    "member_id": {
        "anchor": "INSURED",
        "type": "text"
    },

    "provider_name": {
        "anchor": "PHYSICIAN",
        "type": "text"
    },

    "date_of_service": {
        "anchor": "SERVICE",
        "type": "date"
    },

    "diagnosis_code": {
        "anchor": "DIAGNOSIS",
        "type": "code"
    },

    "procedure_code": {
        "anchor": "CPT",
        "type": "code"
    },

    "charge_amount": {
        "anchor": "TOTAL",
        "type": "money"
    }
}

In [34]:
def find_anchor(df, keyword):

    rows = df[df["text"].str.upper().str.contains(keyword)]

    if len(rows) == 0:
        print(f"{keyword} anchor NOT FOUND")
        return None

    anchor = rows.iloc[0]

    print("\n--- Anchor Found ---")
    print(anchor[["text","left","top"]])

    return anchor

In [35]:
def extract_region(df, anchor):

    x = anchor["left"]
    y = anchor["top"]

    print(f"\nSearching region near X:{x} Y:{y}")

    region = df[
        (df["left"] > x - 50) &
        (df["left"] < x + 900) &
        (df["top"] > y + 30) &
        (df["top"] < y + 150)
    ]

    print("\nCaptured Words:")
    print(region[["text","left","top"]])

    words = region.sort_values(["top","left"])["text"]

    return " ".join(words)

In [36]:
def postprocess(value, field_type):

    if value is None:
        return None

    if field_type == "date":
        m = re.search(r"\d{2}[/\-]\d{2}[/\-]\d{2,4}", value)
        return m.group(0) if m else None

    if field_type == "money":
        m = re.search(r"\d+\.\d{2}", value)
        return m.group(0) if m else None

    if field_type == "code":
        codes = re.findall(r"[A-Z0-9\.]{3,7}", value)
        return codes

    return value.strip()

In [37]:
def extract_claim(df):

    results = {}

    for field, config in FIELD_CONFIG.items():

        print(f"\n========= Extracting {field} =========")

        anchor = find_anchor(df, config["anchor"])

        if anchor is None:
            results[field] = None
            continue

        raw_value = extract_region(df, anchor)

        results[field] = postprocess(
            raw_value,
            config["type"]
        )

    return results

In [38]:
claim_data = extract_claim(ocr_df)

print("\nFINAL OUTPUT:")
print(claim_data)


========= Extracting patient_name =========

--- Anchor Found ---
text    PATIENT
left        214
top        1019
Name: 138, dtype: object

Searching region near X:214 Y:1019

Captured Words:
      text  left   top
163  SMITH   174  1108
164   JOHN  1078  1112

========= Extracting member_id =========

--- Anchor Found ---
text    INSURED’S
left         3150
top           833
Name: 83, dtype: object

Searching region near X:3150 Y:833

Captured Words:
    text  left  top
124    3  3116  910
125    3  3176  910
126    3  3236  910
127    3  3296  910
128    3  3356  910
129    3  3416  910
130    3  3476  910
131    3  3536  910

========= Extracting provider_name =========

--- Anchor Found ---
text    physician
left         3681
top          2883
Name: 498, dtype: object

Searching region near X:3681 Y:2883

Captured Words:
     text  left   top
532    ON  4012  3013
533  FILE  4198  3013

========= Extracting date_of_service =========

--- Anchor Found ---
text    services
left     